# The closed vertex

In [1]:
load("PT_CY5.sage")
import time

## Our goal
The goal is to verify a closed formula for the PT series
$$ \mathsf{PT}_{(d_1,d_2,d_3)}(Z, \mathbb{C}^{\times}_q,\mathsf{T}')$$
of the so-called closed vertex times the affine plane $Z=X\times \mathbb{C}^2$ in low degree. The formula is stated in Conjecture 2.11. We will test the formula for $d_1+d_2+d_3 \leq d_{\mathrm{max}}$ with

In [2]:
dmax = 3

and for a precision

In [3]:
qprec = 3

in the box-counting variable $q$.

## Setup of the PT series
First, let us introduce a function `PT_clvert(d1,d2,d3,qprec)` which determines this PT series $ \mathsf{PT}_{(d_1,d_2,d_3)}(Z, \mathbb{C}^{\times}_q,\mathsf{T}')$. This is a Laurent series in $q$ with coefficients rational functions in $q_1,q_2,q_3$. We set up the ring for our calculation by calling

In [4]:
setup_ring('q1,q2,q3')

We choose the $\mathsf{T}'$ characters `chars` at the four fixed points of the GKM graph as

In [5]:
kappa = 1/(q1*q2*q3)
chars = [
    [1/(q1^2 * kappa), 1/(q3^2 * kappa), 1/(q2^2 * kappa), 1, kappa],
    [q1^2 * kappa, q3^2, q2^2, 1, kappa],
    [q3^2 * kappa, q2^2, q1^2, 1, kappa],
    [q2^2 * kappa, q1^2, q3^2, 1, kappa]
]

Note here that we doubled the weights of the GKM graph presented in the paper. This is to avoid fractional exponents. We will also require the following permutations of `chars[0]` as arguments for edge terms:

In [6]:
chars01 = [1/(q3^2 * kappa), 1/(q2^2 * kappa), 1/(q1^2 * kappa), 1, kappa]
chars02 = [1/(q2^2 * kappa), 1/(q1^2 * kappa), 1/(q3^2 * kappa), 1, kappa]

The normal bundles of the three torus invariant $\mathbb{P}^1$s split into line bundles of the following degrees:

In [7]:
LBdegs = [-1, -1, 0, 0]

We can now define the function `PT_clvert(d1,d2,d3,qprec)`:

In [8]:
def PT_clvert(d1, d2, d3, qprec):
    return sum(
        V([mu1, mu3, mu2], chars[0], qprec) *
        V([mu1.conjugate(), [], []], chars[1], qprec) *
        V([mu3.conjugate(), [], []], chars[2], qprec) *
        V([mu2.conjugate(), [], []], chars[3], qprec) *
        E(mu1, LBdegs, chars[0]) *
        E(mu3, LBdegs, chars01) *
        E(mu2, LBdegs, chars02)
        for mu1 in Partitions(d1)
        for mu2 in Partitions(d2)
        for mu3 in Partitions(d3)
    )

## The conjectural formula
The conjectural formula is presented as a plethystic exponential. Since we would only like to verify the formula in multi-degree $(d_1,d_2,d_3)$ with $d_1+d_2+d_3 \leq d_{\mathrm{max}}$, we evalute the plethystic exponential up to this order and extract conjectural formulae for each individual PT series. First we set up the ring in which the plethystic exponential will take values in:

In [9]:
wR2.<qq1,qq2,qq3,qq4> = LaurentPolynomialRing(QQ)
QR2.<Q1,Q2,Q3> = PowerSeriesRing(wR2.fraction_field(), default_prec = dmax + 1)

We choose to compute the plethystic exponential in the ring
$$ \mathbb{Q}(q_1,q_2,q_3,q_4)[\![Q_1,Q_2,Q_3]\!]$$
first and only later perform the Laurent expansion in $q_4 = q$ in order to increase performance. We define the following auxiliary variables and the function $[t] = t - t^{-1}$ which we denote `br(t)`:

In [10]:
kkappa = 1/(qq1*qq2*qq3)
qq5 = kkappa / qq4

def br(t):
    return t - t^(-1)

The following function evaluates the plethystic exponential which according to Conjecture 2.11 equates to the PT series of the closed vertex:

In [11]:
def PT_clvert_conj_series(dmax):
    Omega100 = 1 / (br(qq4) * br(qq5))
    Omega110 = - br(qq3^2*kkappa) / (br(qq3^2) * br(qq4) * br(qq5))
    Omega101 = - br(qq2^2*kkappa) / (br(qq2^2) * br(qq4) * br(qq5))
    Omega011 = - br(qq1^2*kkappa) / (br(qq1^2) * br(qq4) * br(qq5))
    Omega111 = 1 / (br(qq4) * br(qq5))
    
    return Exp(
               Omega100 * (Q1 + Q2 + Q3) +
               Omega110 * Q1 * Q2 +
               Omega101 * Q1 * Q3 +
               Omega011 * Q2 * Q3 +
               Omega111 * Q1 * Q2 * Q3
    , dmax)

Let us evaluate the conjectural expression for the PT series for all multi-degrees $(d_1,d_2,d_3)$ with $d_1+d_2+d_3 \leq d_{\mathrm{max}}$. We record the formulae in a dictionary `PT_clvert_conj_dict`.

In [12]:
start = time.time()  
PT_clvert_conj_dict = PT_clvert_conj_series(dmax).coefficients()
end = time.time()
print("Expansion of conjectural formula took {:.2f}min.\nNow we \
      check the formulae against PT series.\n".format((end - start)/60.))

Expansion of conjectural formula took 0.55min.
Now we       check the formulae against PT series.



The conjectural formulae for the PT series are still elements of the ring
$$ \mathbb{Q}(q_1,q_2,q_3,q_4)[\![Q_1,Q_2,Q_3]\!]$$
We perform a Laurent expansion in $q_4=q$ up order `qprec` using the following function:

In [13]:
def q_expand(f, qprec):
    f_expanded = Hom(wR2,qR)([q1, q2, q3, q]).extend_to_fraction_field()(f)
    return f_expanded + O(q^(f_expanded.valuation() + 2 * qprec))

## Comparison
Now we can systematically check that the PT series computed via `PT_clvert` indeed equate to the conjectural formulae stored in `PT_clvert_conj_dict`:

In [14]:
for d in range(1, dmax+1):
    for part in Partitions(d, max_length=3):
        dList = part + [0] * (3 - len(part))
        
        start = time.time()
        PT = PT_clvert(*dList, qprec)
        conj = q_expand(PT_clvert_conj_dict[Q1^dList[0] * Q2^dList[1] * Q3^dList[2]], qprec)
        conj_holds = (PT - conj).is_zero()
        end = time.time()
        print("[d1, d2, d3] = {}  -->  {}  ({:.2f}min)".format(dList, conj_holds, (end - start) / 60.))

[d1, d2, d3] = [1, 0, 0]  -->  True  (0.00min)
[d1, d2, d3] = [2, 0, 0]  -->  True  (0.00min)
[d1, d2, d3] = [1, 1, 0]  -->  True  (0.00min)
[d1, d2, d3] = [3, 0, 0]  -->  True  (0.00min)
[d1, d2, d3] = [2, 1, 0]  -->  True  (0.00min)
[d1, d2, d3] = [1, 1, 1]  -->  True  (0.00min)
